## Requirements
- Parsed document table with elements in JSON format.
- Serverless compute v4

## Setup

In [0]:
%pip install langchain-text-splitters==1.0 -q
%restart_python

In [0]:
catalog = "workspace"
schema_input = "bronze"
schema_output = "silver"
parsed_table = f"{catalog}.{schema_input}.docs_parsed"
chunked_table = f"{catalog}.{schema_output}.docs_chunked"

## A. Transform Parsed JSON to Clean Text
Objective: convert parsed JSON text into clean, page-separated plain text ready for use with LLMs. Two methods:
1. LLM-powered semantic cleaning: using the built-in `ai_query` to batch process and convert JSON to markdown text. This preserves more document semantics but is more expensive.
2. Fast concatenation: join all text elements into a single plain text string. This is quick and cost-effective but loses some semantic structure (e.g., page headers)

### A1. Load Parsed Document

In [0]:
parsed_df = spark.read.table(parsed_table)
print(f"Loaded parsed documents from: {parsed_table}")
parsed_df.printSchema()

In [0]:
display(parsed_df)

### A2. LLM-Powered Semantic Cleaning with `ai_query`
- Concept: use the SQL function `ai_query` to feed the parsed JSON content to a LLM-hosted endpoint, with instruction of output format in the system prompt to produce markdown text that retains the original semantic elements captured in the JSON.
- Pros: good performing LLMs can retain semantic information and produce high quality markdown
- Cons: 
    - slower
    - costs money
    - LLMs limited by their context window size

In [0]:
from pyspark.sql.functions import expr


# Specify a foundation model available on your workspace
ENDPOINT = "databricks-llama-4-maverick" # use this to handle large number of tokens

# Specify the system prompt:
prompt_prefix = '''You are a helpful assistant. Given a JSON object representing a parsed document (with pages, elements, and metadata), convet the content into clean, readable markdown. Use "== page ==" to separate each page. Preserve important structure such as headers, tables and caption. Do not include any JSON or code blocks in the output - just the clean markdown text.

JSON:

'''

# Apply `ai_query` to batch process the parsed JSON text
transformed_df = (
    parsed_df.withColumn(
        "clean_markdown_text",
        expr(f"""
            ai_query(
                '{ENDPOINT}',
                CONCAT('{prompt_prefix}', CAST(parsed_content AS STRING)),
                responseFormat => '{{"type":"text"}}'
            )
        """)
    )
)
display(transformed_df.select("path", "clean_markdown_text"))

### A3. Fast Plain Text Conversion

In [0]:
# Helper function to extract the parsed elements' data

import json
from typing import Any, Optional

def _page_id_from_bbox(bbox: Any) -> Optional[int]:
    """
    bbox can be a list[dict], a dict, or None. Return bbox.page_id if present.
    """
    if not bbox:
        return None
    if isinstance(bbox, list) and bbox:
        first = bbox[0] or {}
        return first.get("page_id")
    if isinstance(bbox, dict):
        return bbox.get("page_id")
    return None

def extract_contents_from_json(json_str: str) -> str:
    """
    - Concat element 'content' (fallback to 'description' if missing).
    - Insert '== page ==' when page_id changes.
    - Insert an extra newline after elements whose type != 'text'.
    - Return an error string on failure for easy debugging in DataFrame.
    
    """
    try:
        doc = json.loads(json_str) if isinstance(json_str, str) else json_str
        if not isinstance(doc, dict):
            return ""
        # Support both {"document":{"elements":[...]}} and {"elements":[...]}
        document = doc.get("document", doc)
        elements = document.get("elements", []) if isinstance(document, dict) else []
        if not isinstance(elements, list):
            return ""
        
        out_lines = []
        current_page = None

        for el in elements:
            if not isinstance(el, dict):
                continue

            # Page divider on change
            pid = _page_id_from_bbox(el.get("bbox"))
            if pid is not None and current_page is not None and pid != current_page:
                out_lines.append("")
                out_lines.append("== page ==")
                out_lines.append("")
            if pid is not None:
                current_page = pid

            # Content (fallback to description)
            c = el.get("content")
            if not (isinstance(c, str) and c.strip()):
                c = el.get("description")
            if isinstance(c, str) and c.strip():
                out_lines.append(c)

                # Extra newline after non-tet elements
                t = (el.get("type") or "").lower()
                if t!="text":
                    out_lines.append("") # pruduces a blank line after line joining
        return "\n".join(out_lines)
    
    except Exception as e:
        print(f"Error parsing JSON: {e}")
        

def extract_contents_udf():
    from pyspark.sql.types import StringType
    from pyspark.sql.functions import udf
    @udf(StringType())
    def _udf(json_str):
        try:
            return extract_contents_from_json(json_str)
        except Exception as e:
            print(f"Error parsing JSON: {e}")
            raise e
    return _udf

In [0]:
from pyspark.sql import functions as F

# convert VARIANT/structure/map to a JSON string first (avoids VariantVal issues)
safe_json_col = F.coalesce(
    F.to_json(F.col("parsed_content")),
    F.col("parsed_content").cast("string")
)

# Apply the custom UDF
plain_text_df = parsed_df.withColumn(
    "plain_text",
    extract_contents_udf()(safe_json_col)
)

display(plain_text_df.select("path", "plain_text"))

## B. Chunk Cleaned Text for Retrieval
- Objective: chunk clean, page-separated text
- The text will be split by the page delimiter `== page ==` token.
- Text splitter is LangChain's `RecursiveTextSplitter. This utility automatically handles chunking and overlap.
- Overlap is only introduced when a single input is split into multiple chunks, i.e., if an input text is > the chunk size.
- In this example, if a page contains more than the chunk size of 2000 characters, it will be split and the chunks will have chunk overlaps. Else the whole page will fit in 1 chunk.
- Some overlap strategies for long chunks to provide more context of what happens before and after each chunk:
    - use previous para as overlap
    - use the summary of the previous section(s) as the overlap

In [0]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pyspark.sql.types import StructType, StructField, StringType
import pandas as pd


CHUNK_SIZE = 2000
CHUNK_OVERLAP = 200

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n== page ==\n", "== page ==", "\n\n", "\n", " ", ""]
)

# Define the output schema for the chunked DF
# Each row contains the document path and a single text chunk
schema = StructType([
    StructField("path", StringType(), True),
    StructField("chunk", StringType(), True)
])

def split_rows(iterator):
    """
    mapInPandas function: input pdfs with columns [path, plain_text],
    output rows [path, chunk].
    This functino splits each document text into chunks and yields them for DataFrame construction.
    """
    for pdf in iterator:
        out = []
        for _, row in pdf.iterrows():
            path = row["path"]
            text = row["plain_text"]
            if isinstance(text, str) and text.strip():
                for c in splitter.split_text(text):
                    if c and c.strip():
                        out.append((path, c))
        yield pd.DataFrame(out, columns=["path", "chunk"])

# Apply the splitter to the plain text dataframe
df_chunks = (
    plain_text_df
    .select("path", "plain_text")
    .mapInPandas(split_rows, schema=schema)
)

display(df_chunks)


## C. Save Chunked Data to Delta Table for Future Embedding Generation

In [0]:
from pyspark.sql import functions as F


# add incremental ID to each chunk
df_chunks = df_chunks.withColumn("id", F.monotonically_increasing_id())

# save the chunked data to table
df_chunks.write.format("delta").mode("overwrite").option(
    "mergeSchema", "true"
).saveAsTable(chunked_table)

display(spark.read.table(chunked_table))